In [1]:
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer, util
import torch
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

/home/le/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Semantic Search Implementation

In [2]:
def validate_and_save_results(seed_dict: dict, candidate_dict: dict, model_name: str, output_filepath: str):
    """
    Validates AI-generated keywords against seed keywords, ensuring the
    comparison happens only WITHIN the same category.

    Args:
        seed_dict: Dictionary of {category: [seed_keywords]}.
        candidate_dict: Dictionary of {category: [ai_generated_keywords]}.
        model_name: Name of the SentenceTransformer model to use.
        output_filepath: Path to save the validation results as a CSV file.
    """
    model = SentenceTransformer(model_name) 
    print("--- Starting Intra-Category Validation ---")

    # Initialize a list to store validation results
    validation_results = []

    # Loop through each category from your original seed keyword dictionary
    for category, seed_keywords in seed_dict.items():
        print(f"\n\n--- Processing Category: '{category}' ---")

        # 1. Check if the AI dictionary also has this category
        if category not in candidate_dict or not candidate_dict[category]:
            print("  - No AI-generated keywords found for this category to compare against.")
            continue

        # 2. This is the key change: The "corpus" is now only the AI keywords
        #    from the *current* category.
        ai_keywords_for_this_category = candidate_dict[category]

        # 3. Encode both lists for the current category
        # Queries = seed keywords for this category
        query_embeddings = model.encode(seed_keywords, convert_to_tensor=True)
        query_embeddings = query_embeddings.to("cuda")
        query_embeddings = util.normalize_embeddings(query_embeddings)
        # Corpus = AI-generated keywords for this category
        corpus_embeddings = model.encode(ai_keywords_for_this_category, convert_to_tensor=True)
        corpus_embeddings = corpus_embeddings.to("cuda")
        corpus_embeddings = util.normalize_embeddings(corpus_embeddings)

        # 4. Perform the semantic search within the category
        # Find the top 50 most similar AI keywords for each seed keyword
        search_results = util.semantic_search(query_embeddings, corpus_embeddings, top_k=50, score_function=util.dot_score)
        # dot_score is used for cosine similarity since we normalized the embeddings and also for speed (from 10min to 6.5s with BGE-M3)

        for i, keyword in enumerate(seed_keywords):
            hits = search_results[i]
            if not hits:
                continue
            for hit in hits:
                # For each hit, create a dictionary and append it to our results list
                result_entry = {
                    'category': category,
                    
                    'generated_keyword': ai_keywords_for_this_category[hit['corpus_id']],
                    'similarity_score': hit['score']
                }
                validation_results.append(result_entry)

    # Convert the results list to a DataFrame
    validation_df = pd.DataFrame(validation_results)

    # Drop duplicates based on 'Category' and 'AI-Generated Keyword'
    validation_df = validation_df.drop_duplicates(subset=['category', 'generated_keyword'])

    # Save the results to a CSV file
    validation_df.to_csv(output_filepath, index=False)
    print(f"Validation results saved to {output_filepath}")
    return validation_df
            


In [3]:
# Load seed keywords from taxonomy_sheet_v2.csv (7 categories)
taxonomy_df = pd.read_csv('data/00_raw/taxonomy_sheet_v2.csv')

# Parse seed keywords from taxonomy
seed_keywords_dict = {}
for _, row in taxonomy_df.iterrows():
    category = row['category']
    keywords_str = row['keywords']
    # Split by newlines and clean
    keywords = [kw.strip() for kw in keywords_str.split('\n') if kw.strip()]
    seed_keywords_dict[category] = keywords

seed_keyword_count = sum(len(kws) for kws in seed_keywords_dict.values())

# Load the extracted candidate phrases (v4 - enhanced boundaries)
candidate_phrases = pd.read_csv('results/phase i/11_candidate_phrases_v4.csv')

# Group candidate phrases by category
candidate_synonyms_dict = candidate_phrases.groupby('category')['candidate_phrase'].apply(list).to_dict()

print(f"📊 Loaded {seed_keyword_count} seed keywords from {len(seed_keywords_dict)} categories")
print(f"📊 Loaded {len(candidate_phrases)} candidate phrases from {len(candidate_synonyms_dict)} categories")
print(f"\n🔄 Calculating intra-category semantic similarities using BGE-M3...")

📊 Loaded 74 seed keywords from 7 categories
📊 Loaded 1400 candidate phrases from 7 categories

🔄 Calculating intra-category semantic similarities using BGE-M3...


In [4]:
output_filename = 'results/phase i/12_keyword_validation_results_v4.csv'
validation_df = validate_and_save_results(seed_keywords_dict, candidate_synonyms_dict, 'BAAI/bge-m3', output_filename) 
# bge-m3 is better than all-MiniLM-L6-v2, especially for extracting similar concepts, not necessarily exact synonyms
print(f"\n✅ Similarity calculation complete!")
print(f"📁 Results saved to: {output_filename}")

--- Starting Intra-Category Validation ---


--- Processing Category: 'Governance' ---


--- Processing Category: 'Personnel' ---


--- Processing Category: 'Products' ---


--- Processing Category: 'IT/Data' ---


--- Processing Category: 'Processes' ---


--- Processing Category: 'Legal' ---


--- Processing Category: 'Communication' ---
Validation results saved to results/phase i/12_keyword_validation_results_v4.csv

✅ Similarity calculation complete!
📁 Results saved to: results/phase i/12_keyword_validation_results_v4.csv
